# Amendment 26: the entangling-range control (experiment 31)

Three arms, 40 seeds, 4 datasets, n = 5 to 100, d = 4: `quantum_vqc`, `quantum_vqc_range1`, `quantum_reupload`.

**Before you start:** Amendment 26 must be committed and pushed. Put that commit's SHA in cell 1.

**Runtime:** use the **CPU** runtime (High-RAM if offered) in every session. The notebook stops if you mix device types.

**Order:**
1. Run cells 1 to 9. Each one must end with `OK`. If a cell fails, stop and send me its output.
2. Cell 10 is the pilot. It prints the time estimate and a suggested number of sessions.
3. Cell 11 starts the real run for this session's seeds. Cell 12 waits and shows progress.
4. After **all** sessions have finished, run cell 13 (analysis) once, in any session.

If a session disconnects: open a new one, run cells 1 to 9 again, then cell 11 with the same `SESSION_INDEX`. Finished runs are skipped.

In [ ]:
# 1. SETTINGS. Edit these three lines only.
EXPECTED_SHA = "PASTE_THE_FULL_SHA_OF_YOUR_AMENDMENT_26_COMMIT"
N_SESSIONS   = 1        # how many Colab sessions share the work (cell 10 suggests a number)
SESSION_INDEX = 0       # 0, 1, ... N_SESSIONS-1: which share this session runs

BACKUP = "/content/drive/MyDrive/QML-backup-run"      # the Drive folder with the 10 October backups
EXP    = BACKUP + "/exp31"                              # results of this experiment go here
REPO   = "/content/repo"
ART    = "/content/art"                                 # QML_ARTIFACT_ROOT for this session
ENV    = "/content/env"
PY     = ENV + "/bin/python"
NS, PILOT = "31_entangling_range", "31_pilot"
ARMS   = ["quantum_vqc", "quantum_vqc_range1", "quantum_reupload"]
DATASETS = ["breastmnist", "pneumoniamnist", "bloodmnist", "pathmnist"]
REGIMES  = ["5", "10", "20", "50", "100"]

import os, subprocess, json, glob, time, shutil
os.environ["QML_ARTIFACT_ROOT"] = ART
os.environ["MPLBACKEND"] = "Agg"

def sh(cmd, log=None, check=True, env=None):
    """Run a bash command from the repo, print its output, append it to a log on Drive."""
    e = dict(os.environ, **(env or {}))
    p = subprocess.Popen(["bash", "-lc", cmd], cwd=REPO if os.path.isdir(REPO) else "/content",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=e)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    rc = p.wait()
    if log:
        os.makedirs(EXP + "/logs", exist_ok=True)
        with open(f"{EXP}/logs/{log}", "a") as f:
            f.write(f"\n$ {cmd}\n" + "".join(lines) + f"exit code {rc}\n")
    if check and rc != 0:
        raise RuntimeError(f"FAILED (exit {rc}): {cmd}")
    return "".join(lines)

def status():
    recs = []
    for fn in glob.glob(f"{EXP}/shards/{NS}/*.json"):
        try:
            recs.append(json.load(open(fn)))
        except Exception:
            pass
    per_arm = {a: sum(r["keys"]["arm"] == a for r in recs) for a in ARMS}
    shas = sorted({r.get("git_sha", "?")[:8] for r in recs})
    preds = len(glob.glob(f"{EXP}/predictions/{NS}/*.npz"))
    print(time.strftime("%H:%M"), "| shards per arm (all sessions, of 800):", per_arm,
          "| predictions:", preds, "| git SHAs:", shas, flush=True)
    return per_arm, shas, recs

assert 0 <= SESSION_INDEX < N_SESSIONS
print("OK")

In [ ]:
# 2. MOUNT DRIVE and check the backups are there.
from google.colab import drive
drive.mount("/content/drive")
fc = sorted(glob.glob(BACKUP + "/feature_cache_*.tgz"))
rs = sorted(glob.glob(BACKUP + "/results_*.tgz"))
print("feature cache backup:", fc)
print("results backup:      ", rs)
assert fc, "feature_cache_*.tgz not found in " + BACKUP
os.makedirs(EXP + "/logs", exist_ok=True)
print("OK")

In [ ]:
# 3. CLONE THE REPO at the pushed commit and check that it is clean and carries Amendment 26.
if not os.path.isdir(REPO):
    sh(f"git clone -b feature/journal-expansion https://github.com/anishfelixm/quantum-classical-expressivity {REPO}")
sh(f"git fetch origin && git checkout --detach {EXPECTED_SHA}")
head = sh("git rev-parse HEAD").strip()
assert head == EXPECTED_SHA, f"HEAD is {head}, expected {EXPECTED_SHA}"
assert sh("git status --porcelain").strip() == "", "the clone has local changes"
plan = open(REPO + "/docs/analysis_plan.md").read()
assert "## Amendment 26" in plan, "Amendment 26 is not in docs/analysis_plan.md at this commit"
for f in ["tests/test_entangling_ranges.py", "scripts/analyse_entangling_range.py", "scripts/audit_arm_pairs.py"]:
    assert os.path.exists(f"{REPO}/{f}"), f + " missing"
print("HEAD", head)
print("OK")

In [ ]:
# 4. PYTHON ENVIRONMENT: Python 3.10 with the pinned versions of requirements.lock.txt (about 5 minutes).
if not os.path.exists(PY):
    sh("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba")
    sh(f"/content/bin/micromamba create -y -q -p {ENV} -c conda-forge python=3.10 pip")
    sh(f"{PY} -m pip install -q -r requirements.lock.txt pytest "
       f"--extra-index-url https://download.pytorch.org/whl/cu118")
sh(f"{PY} -c \"import torch, pennylane, numpy; print('torch', torch.__version__, '| pennylane', pennylane.__version__, '| numpy', numpy.__version__)\"",
   log="gates.txt")
print("OK")

In [ ]:
# 5. DEVICE LOCK. The first session records its device; every later session must match it.
#    Arms of the same seed always run in the same session, so every paired difference is on one machine.
info = sh(f"{PY} -c \"import sys; sys.path.insert(0,'src'); import config, torch; "
          f"print(config.DEVICE.type); print(torch.get_num_threads())\"").split()
dev, threads = info[-2], int(info[-1])
cpu = sh("lscpu | grep 'Model name' | sed 's/.*: *//'", check=False).strip()
cores = os.cpu_count()
lock = EXP + "/device_lock.json"
if os.path.exists(lock):
    want = json.load(open(lock))["device"]
    assert dev == want, f"this session is on {dev}, but earlier sessions used {want}. Change the runtime type."
else:
    assert dev == "cpu", "Amendment 26 runs on the CPU runtime. Change the runtime type to CPU."
    json.dump({"device": dev, "first_session_cpu": cpu, "date": time.strftime("%Y-%m-%d %H:%M")}, open(lock, "w"))
with open(EXP + "/logs/sessions.txt", "a") as f:
    f.write(f"{time.strftime('%Y-%m-%d %H:%M')} session {SESSION_INDEX}/{N_SESSIONS} device={dev} cpu='{cpu}' cores={cores} sha={EXPECTED_SHA[:8]}\n")
print(f"device={dev}  cpu='{cpu}'  cores={cores}")
print("OK")

In [ ]:
# 6. DRAW THE THREE CIRCUITS at d = 4 (kept in logs/circuits.txt).
#    Expected CNOT pairs, second layer: quantum_vqc 0-2 1-3 2-0 3-1; the other two 0-1 1-2 2-3 3-0.
draw = r"""
import sys, torch, pennylane as qml
sys.path.insert(0, 'src'); sys.path.insert(0, 'scripts')
from models.registry import build_arm
from audit_arm_pairs import circuit_profile
for arm in ['quantum_vqc', 'quantum_vqc_range1', 'quantum_reupload']:
    h = build_arm(arm, d=4, num_classes=2, seed=42, build_backbone=False).head
    x = torch.zeros(4, dtype=torch.float64); w = h.q_layer.weights.detach().double()
    print('=' * 30, arm)
    print(qml.draw(h.q_layer.qnode, level='device', decimals=None, max_length=250)(x, w))
    p = circuit_profile(h)
    print('encodings:', p['encodings'], '| CNOT pairs in order:', p['entangling pattern'])
"""
open("/content/draw.py", "w").write(draw)
sh(f"{PY} /content/draw.py", log="circuits.txt")
print("OK")

In [ ]:
# 7. TESTS: the new range tests, parameter parity, and the two function-class tests.
sh(f"{PY} -m pytest -q -s tests/test_entangling_ranges.py tests/test_parity.py "
   f"tests/test_fourier_equivalence.py tests/test_reupload_spectrum.py", log="gates.txt")
print("OK")

In [ ]:
# 8. PARITY (project rule) and the ARM-PAIR AUDIT. Both must pass.
parity = r"""
import sys; sys.path.insert(0, 'src')
from models.registry import build_arm, count_head_params
bad = 0
for arm, kw in [('quantum_vqc', {}), ('matched_param_fullrank', {}), ('low_rank', {'head_rank': 2}),
                ('quantum_vqc_range1', {}), ('quantum_reupload', {})]:
    n = count_head_params(build_arm(arm, d=4, num_classes=2, seed=42, build_backbone=False, **kw))['head']
    print(f'{arm:24s} {kw}  head parameters at d=4: {n}')
    bad += (n != 24)
sys.exit(1 if bad else 0)
"""
open("/content/parity.py", "w").write(parity)
sh(f"{PY} /content/parity.py", log="gates.txt")
sh(f"{PY} scripts/audit_arm_pairs.py", log="audit.txt")
print("OK")

In [ ]:
# 9. ARTIFACT FOLDER and WARM FEATURE CACHE.
#    shards/ and predictions/ for experiment 31 live on Drive; the feature cache is local.
#    Every one of the 800 cache files must exist BEFORE training, so no run builds its own
#    cache (a run that builds it draws its initialisation differently).
for sub in ("shards", "predictions"):
    os.makedirs(f"{ART}/{sub}", exist_ok=True)
    for ns in (NS, PILOT):
        os.makedirs(f"{EXP}/{sub}/{ns}", exist_ok=True)
        link = f"{ART}/{sub}/{ns}"
        if not os.path.islink(link):
            os.symlink(f"{EXP}/{sub}/{ns}", link)

# 9a. restore the 10 October feature cache
FC = f"{ART}/feature_cache"
if not os.path.islink(FC) and not os.path.isdir(FC):
    os.makedirs("/content/fc_extract", exist_ok=True)
    sh(f"tar -xzf '{fc[-1]}' -C /content/fc_extract")
    pts = glob.glob("/content/fc_extract/**/*__r*__s*.pt", recursive=True)
    assert pts, "no cache files found inside the backup"
    os.symlink(os.path.dirname(pts[0]), FC)
print("feature cache folder:", os.path.realpath(FC))

# 9b. the recorded 01_frozen_tuned shards, for the descriptive device check only
REC = f"{ART}/shards/01_frozen_tuned"
if rs and not os.path.exists(REC):
    os.makedirs("/content/rs_extract", exist_ok=True)
    sh(f"tar -xzf '{rs[-1]}' -C /content/rs_extract --wildcards '*shards/01_frozen_tuned/*'", check=False)
    hits = glob.glob("/content/rs_extract/**/shards/01_frozen_tuned", recursive=True)
    if hits:
        os.symlink(hits[0], REC)
print("recorded 01_frozen_tuned shards:", len(glob.glob(REC + "/*.json")) if os.path.exists(REC) else "not found (device check will be skipped)")

# 9c. check all 800 cache files; build any missing ones in a SEPARATE process
seeds_all = json.loads(sh(f"{PY} -c \"import sys, json; sys.path.insert(0,'src'); import config; "
                          f"print(json.dumps(config.CONFIRMATORY_SEEDS))\"").strip().splitlines()[-1])
assert len(seeds_all) == 40
need = [(d, r, s) for d in DATASETS for r in REGIMES for s in seeds_all]
missing = [t for t in need if not os.path.exists(f"{FC}/{t[0]}__r{t[1]}__s{t[2]}.pt")]
print(f"cache files present: {len(need) - len(missing)} of {len(need)}")
if missing:
    json.dump(missing, open("/content/missing.json", "w"))
    warm = r"""
import sys, json, importlib; sys.path.insert(0, 'src')
m = importlib.import_module('01_frozen_backbone_ablation')
for d, r, s in json.load(open('/content/missing.json')):
    m.get_cached_features(d, r, s); print('built', d, r, s, flush=True)
"""
    open("/content/warm.py", "w").write(warm)
    sh(f"{PY} /content/warm.py", log="cache_prewarm.txt")
    missing = [t for t in need if not os.path.exists(f"{FC}/{t[0]}__r{t[1]}__s{t[2]}.pt")]
assert not missing, f"{len(missing)} cache files still missing"
print("OK")

In [ ]:
# 10. PILOT: time one PathMNIST run per arm at n = 100 and at n = 5 (seed 42, namespace 31_pilot,
#     never analysed). The three arms run side by side, exactly as in cell 11, so the timing is honest.
#     Estimate per session = sum over its runs of (a + b * images), with a and b fitted to the two
#     PathMNIST points. Assumption: run time grows linearly with training + validation images, and the
#     fixed cost per run is PathMNIST's (the largest test set), so the estimate leans high.
THREADS = max(1, cores // len(ARMS))
TENV = {"OMP_NUM_THREADS": str(THREADS), "MKL_NUM_THREADS": str(THREADS)}
def pilot_cmd(arm, regime):
    return (f"{PY} -u src/01_frozen_backbone_ablation.py --arms {arm} --dims 4 --datasets pathmnist "
            f"--regimes {regime} --seeds 42 --lr-head 1e-2 --lr-quantum 1e-2 --experiment {PILOT}")
for sub in ("shards", "predictions"):          # scratch namespace: clear it so every pilot run really trains
    for f in glob.glob(f"{EXP}/{sub}/{PILOT}/*"):
        os.remove(f)
times = {}
for regime in ("100", "5"):
    t0, procs = time.time(), {}
    for arm in ARMS:
        procs[arm] = subprocess.Popen(["bash", "-lc", pilot_cmd(arm, regime) + f" >> {EXP}/logs/pilot_{arm}.txt 2>&1"],
                                      cwd=REPO, env=dict(os.environ, **TENV))
    for arm, p in procs.items():
        assert p.wait() == 0, f"pilot failed for {arm}: see {EXP}/logs/pilot_{arm}.txt"
        times[(arm, regime)] = time.time() - t0
    print(f"PathMNIST n={regime}: " + ", ".join(f"{a} {times[(a, regime)]/60:.1f} min" for a in ARMS))

size_py = r"""
import sys, json, torch
fc, seed = sys.argv[1], int(sys.argv[2]); out = {}
for d in ['breastmnist','pneumoniamnist','bloodmnist','pathmnist']:
    for r in ['5','10','20','50','100']:
        b = torch.load(f'{fc}/{d}__r{r}__s{seed}.pt', map_location='cpu', weights_only=True)
        out[f'{d}|{r}'] = int(b['train_x'].shape[0] + b['val_x'].shape[0])
print(json.dumps(out))
"""
open("/content/size.py", "w").write(size_py)
sizes = json.loads(sh(f"{PY} /content/size.py {FC} 42").strip().splitlines()[-1])
s100, s5 = sizes["pathmnist|100"], sizes["pathmnist|5"]
worst = 0.0
for arm in ARMS:
    b = max((times[(arm, "100")] - times[(arm, "5")]) / (s100 - s5), 0.0)
    a = max(times[(arm, "5")] - b * s5, 0.0)
    per_seed = sum(a + b * n for n in sizes.values())
    worst = max(worst, per_seed)
    print(f"{arm:20s} a={a:6.1f}s  b={b*1000:6.2f}ms/image  ->  {per_seed/3600:.2f} h per seed (20 cells)")
total_h = worst * 40 / 3600
print(f"\nOne session running all 40 seeds: about {total_h:.1f} h (slowest arm sets the pace).")
for n in (1, 2, 3, 4, 5):
    print(f"  {n} session(s): about {total_h/n:.1f} h each")
print("Pick N_SESSIONS so each session stays under about 20 h, set it in cell 1 in EVERY session.")
with open(EXP + "/logs/pilot_estimate.txt", "a") as f:
    f.write(json.dumps({"date": time.strftime("%Y-%m-%d %H:%M"), "cpu": cpu, "cores": cores, "threads": THREADS,
                        "times_s": {f"{a}|{r}": t for (a, r), t in times.items()}, "sizes": sizes,
                        "total_hours_one_session": total_h}) + "\n")
print("OK")

In [ ]:
# 11. START THE RUN for this session's seeds: all three arms, side by side, in the background.
#     Same commit, same device type, same cache, same rate for every arm. Runs already on Drive are skipped.
assert sh("git rev-parse HEAD").strip() == EXPECTED_SHA and sh("git status --porcelain").strip() == ""
THREADS = max(1, cores // len(ARMS))
TENV = {"OMP_NUM_THREADS": str(THREADS), "MKL_NUM_THREADS": str(THREADS)}
my_seeds = seeds_all[SESSION_INDEX::N_SESSIONS]
print(f"session {SESSION_INDEX} of {N_SESSIONS}: {len(my_seeds)} seeds, {len(my_seeds)*20*len(ARMS)} runs")
print("seeds:", my_seeds)
running = {}
for arm in ARMS:
    cmd = (f"{PY} -u src/01_frozen_backbone_ablation.py --arms {arm} --dims 4 "
           f"--datasets {' '.join(DATASETS)} --regimes {' '.join(REGIMES)} "
           f"--seeds {' '.join(map(str, my_seeds))} --lr-head 1e-2 --lr-quantum 1e-2 --experiment {NS}")
    log = f"{EXP}/logs/run_s{SESSION_INDEX}_{arm}.txt"
    running[arm] = subprocess.Popen(["bash", "-lc", f"{cmd} >> {log} 2>&1"], cwd=REPO,
                                    env=dict(os.environ, **TENV), start_new_session=True)
    print(f"started {arm} (pid {running[arm].pid}), log: {log}")
print("OK")

In [ ]:
# 12. WAIT AND WATCH. Prints progress every 15 minutes until this session's three processes end.
#     Keep this cell running and the tab open, so Colab does not disconnect.
while True:
    status()
    alive = {a: p.poll() is None for a, p in running.items()}
    if not any(alive.values()):
        break
    time.sleep(900)
codes = {a: p.returncode for a, p in running.items()}
print("exit codes:", codes)
assert all(c == 0 for c in codes.values()), "a process failed: see its run_s*_<arm>.txt log on Drive"
print("this session is done. OK")

In [ ]:
# 13. ANALYSIS. Run ONCE, after ALL sessions are done. Checks integrity first, then runs the
#     pre-registered exploratory analysis (B = 2000) and copies the results to Drive.
per_arm, shas, recs = status()
assert all(v == 800 for v in per_arm.values()), f"not complete: {per_arm}"
assert len(shas) == 1 and EXPECTED_SHA.startswith(shas[0]), f"more than one commit in the shards: {shas}"
no_pred = [r["keys"] for r in recs
           if not os.path.exists(f"{EXP}/predictions/{NS}/" + os.path.basename(r.get("predictions_file", "none")))]
assert not no_pred, f"{len(no_pred)} runs have no predictions file, e.g. {no_pred[:2]}"
out = sh(f"{PY} -u scripts/analyse_entangling_range.py --workers {max(1, cores - 1)}", log="stats_entangling.txt")
assert "SEED-LEVEL FALLBACK IN USE" not in out, "STOP: seed-level fallback in use. Do not use these numbers."
shutil.copy(f"{ART}/entangling_range.json", f"{EXP}/entangling_range.json")
print("\nResults: " + EXP + "/entangling_range.json and logs/stats_entangling.txt. OK")